In [1]:
import os
import boto3
import duckdb
import pandas as pd
import dataframe_image as dfi
from io import BytesIO
from dotenv import load_dotenv

# 1. Load AWS Credentials from .env
load_dotenv(dotenv_path="../../.env")
AWS_REGION = os.getenv("AWS_REGION")
PROCESSED_BUCKET = 'ra2-processed-data'

# 2. Setup product_insights_output folder
product_output_folder = "../../eda_outputs/product_insights_output"
os.makedirs(product_output_folder, exist_ok=True)

# 3. Connect to S3 securely
s3 = boto3.client('s3', region_name=AWS_REGION)
print("🔗 Connecting to S3 Processed Bucket for Product Insights...\n")

# 4. Download ALL files from S3 and explicitly register them in DuckDB
response = s3.list_objects_v2(Bucket=PROCESSED_BUCKET)

print("📥 Downloading and Registering Clean Tables:")
for obj in response.get('Contents', []):
    file_key = obj['Key']
    
    if file_key.endswith('.csv'):
        table_name = file_key.replace('.csv', '').lower()
        print(f"   -> Fetching {file_key} and registering as table: '{table_name}'...")
        
        csv_obj = s3.get_object(Bucket=PROCESSED_BUCKET, Key=file_key)
        df_temp = pd.read_csv(BytesIO(csv_obj['Body'].read()))
        
        # Explicitly register the dataframe into DuckDB
        duckdb.register(table_name, df_temp)

print("\n✅ All clean data successfully registered in DuckDB for Product Insights!")

# 5. DuckDB Validation Function (Saves to Product Insights Output Folder)
def save_sql_to_image(sql_query, image_name):
    print(f"Executing query for: {image_name}...")
    df = duckdb.query(sql_query).df()
    
    file_path = os.path.join(product_output_folder, f"{image_name}.png")
    dfi.export(df, file_path, table_conversion="matplotlib")
    
    print(f"✅ Image saved successfully at: {file_path}\n")
    return df

c:\Users\pushp\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (7.6.0)/charset_normalizer (3.4.0) doesn't match a supported version!
  warnings.warn(


🔗 Connecting to S3 Processed Bucket for Product Insights...

📥 Downloading and Registering Clean Tables:
   -> Fetching customers.csv and registering as table: 'customers'...
   -> Fetching delivery.csv and registering as table: 'delivery'...
   -> Fetching feedback.csv and registering as table: 'feedback'...
   -> Fetching menu_items.csv and registering as table: 'menu_items'...
   -> Fetching order_items.csv and registering as table: 'order_items'...
   -> Fetching orders.csv and registering as table: 'orders'...
   -> Fetching outlets.csv and registering as table: 'outlets'...
   -> Fetching promotions.csv and registering as table: 'promotions'...

✅ All clean data successfully registered in DuckDB for Product Insights!
